In [ ]:
!pip install iterative-stratification

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import log_loss

In [11]:
SEED = 6769
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
DATA_DIR = '/kaggle/input/competitions/lish-moa/'
train_features = pd.read_csv(f'{DATA_DIR}train_features.csv')
scored_targets = pd.read_csv(f'{DATA_DIR}train_targets_scored.csv')
train_drug = pd.read_csv(f'{DATA_DIR}train_drug.csv')
test_features = pd.read_csv(f'{DATA_DIR}test_features.csv')
sample_sub = pd.read_csv(f'{DATA_DIR}sample_submission.csv')

train_features["cp_time"] = train_features["cp_time"].astype(str)
test_features["cp_time"] = test_features["cp_time"].astype(str)
target_cols = [c for c in scored_targets.columns if c != "sig_id"]

In [13]:
train_mask = train_features["cp_type"] != "ctl_vehicle"
X_train_full = train_features[train_mask].reset_index(drop=True)
y_train_full = scored_targets[train_mask].reset_index(drop=True)
drug_full = train_drug[train_mask].reset_index(drop=True)

test_mask = test_features["cp_type"] != "ctl_vehicle"
X_test_valid = test_features[test_mask].reset_index(drop=True)

In [14]:
data = y_train_full.merge(drug_full, on="sig_id", how="left")
drug_y = data.groupby("drug_id")[target_cols].max()

fold_of_drug = {}
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (_, val_idx) in enumerate(mskf.split(drug_y, drug_y)):
    for d in drug_y.index[val_idx]:
        fold_of_drug[d] = fold

data["fold"] = data["drug_id"].map(fold_of_drug)
X_train_full["fold"] = data["fold"]

In [15]:
def build_fold_features(tr_df, va_df, te_df, var_threshold=0.95):
    """Applies QuantileTransformer, PCA, and OneHotEncoder strictly fit on training data."""
    g_cols = [c for c in tr_df.columns if c.startswith("g-")]
    c_cols = [c for c in tr_df.columns if c.startswith("c-")]
    
    g_scaler = QuantileTransformer(output_distribution="normal").fit(tr_df[g_cols])
    c_scaler = QuantileTransformer(output_distribution="normal").fit(tr_df[c_cols])
    
    g_tr, g_va, g_te = g_scaler.transform(tr_df[g_cols]), g_scaler.transform(va_df[g_cols]), g_scaler.transform(te_df[g_cols])
    c_tr, c_va, c_te = c_scaler.transform(tr_df[c_cols]), c_scaler.transform(va_df[c_cols]), c_scaler.transform(te_df[c_cols])
    
    def get_pca(X, threshold):
        cum = np.cumsum(PCA(random_state=SEED).fit(X).explained_variance_ratio_)
        n_comp = int(np.searchsorted(cum, threshold) + 1)
        pca = PCA(n_components=n_comp, random_state=SEED).fit(X)
        return pca
        
    g_pca = get_pca(g_tr, var_threshold)
    c_pca = get_pca(c_tr, var_threshold)
    
    mg_tr, mg_va, mg_te = g_pca.transform(g_tr), g_pca.transform(g_va), g_pca.transform(g_te)
    mc_tr, mc_va, mc_te = c_pca.transform(c_tr), c_pca.transform(c_va), c_pca.transform(c_te)
    
    enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(tr_df[["cp_dose", "cp_time"]])
    cat_tr = enc.transform(tr_df[["cp_dose", "cp_time"]])
    cat_va = enc.transform(va_df[["cp_dose", "cp_time"]])
    cat_te = enc.transform(te_df[["cp_dose", "cp_time"]])
    
    def assemble(mg, mc, cat):
        return torch.tensor(np.hstack([mg, mc, cat]), dtype=torch.float32)
        
    return assemble(mg_tr, mc_tr, cat_tr), assemble(mg_va, mc_va, cat_va), assemble(mg_te, mc_te, cat_te)

In [ ]:
def moa_logloss(y_true, y_pred, eps=1e-15):
    y_true = np.asarray(y_true, dtype=float)
    p = np.clip(np.asarray(y_pred, dtype=float), eps, 1 - eps)
    per_cell = -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
    return per_cell.mean(axis=0).mean()

def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy().astype(np.float32)
    return np.asarray(x, dtype=np.float32)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S",
                    handlers=[logging.StreamHandler(sys.stdout)])
log = logging.getLogger("xgb_train")


XGB_PARAMS = dict(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    tree_method="hist", device="cuda",
    eval_metric="logloss", verbosity=0,
)

K = 5

oof_preds = np.zeros((len(X_train_full), len(target_cols)))
test_preds = np.zeros((len(X_test_valid), len(target_cols)))
fold_scores = []
t_start = time.time()

log.info(f"Starting {K}-fold XGBoost training | {len(target_cols)} targets | params={XGB_PARAMS}")

for fold in range(K):
    fold_t0 = time.time()
    log.info(f"===== Fold {fold}/{K-1} =====")

    tr_idx = X_train_full[X_train_full["fold"] != fold].index
    va_idx = X_train_full[X_train_full["fold"] == fold].index
    log.info(f"  train rows: {len(tr_idx)} | val rows: {len(va_idx)}")

    X_tr_df, X_va_df = X_train_full.loc[tr_idx], X_train_full.loc[va_idx]
    y_tr = y_train_full.loc[tr_idx, target_cols].values
    y_va = y_train_full.loc[va_idx, target_cols].values

    X_tr, X_va, X_te = build_fold_features(X_tr_df, X_va_df, X_test_valid)
    X_tr, X_va, X_te = to_numpy(X_tr), to_numpy(X_va), to_numpy(X_te)
    log.info(f"  features built: X_tr {X_tr.shape}, X_va {X_va.shape}, X_te {X_te.shape}")

    p_va = np.zeros_like(y_va, dtype=float)
    p_te = np.zeros((len(X_te), len(target_cols)))
    target_t0 = time.time()

    for j, tgt in enumerate(target_cols):
        y_col = y_tr[:, j]
        if y_col.min() == y_col.max():  
            p_va[:, j] = y_col.mean()
            p_te[:, j] = y_col.mean()
            continue
        model = xgb.XGBClassifier(**XGB_PARAMS)
        model.fit(X_tr, y_col)
        p_va[:, j] = model.predict_proba(X_va)[:, 1]
        p_te[:, j] = model.predict_proba(X_te)[:, 1]

        if (j + 1) % 20 == 0 or j == len(target_cols) - 1:
            log.info(f"    [{j+1}/{len(target_cols)}] {tgt}  ({time.time()-target_t0:.1f}s elapsed)")

    oof_preds[va_idx] = p_va
    test_preds += p_te / K

    fold_score = moa_logloss(y_va, p_va)
    fold_scores.append(fold_score)
    log.info(f"  fold {fold} log loss: {fold_score:.5f}  (fold time: {time.time()-fold_t0:.1f}s)")

cv = moa_logloss(y_train_full[target_cols].values, oof_preds)
log.info("===== Done =====")
log.info(f"CV log loss: {cv:.5f}")
log.info(f"fold mean +- std: {np.mean(fold_scores):.5f} +- {np.std(fold_scores):.5f}")
log.info(f"total time: {time.time()-t_start:.1f}s")




In [17]:
oof_loss = log_loss(y_train_full[target_cols].values.ravel(), oof_preds.ravel(), labels=[0,1])
print(f"\\nOverall OOF Log Loss: {oof_loss:.5f}")

\nOverall OOF Log Loss: 0.02371


In [19]:
sub = sample_sub.copy()
sub[target_cols] = 0.0 
sub.loc[test_mask, target_cols] = test_preds
sub.to_csv("submission.csv", index=False)
print("submission.csv successfully saved.")

submission.csv successfully saved.
